In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random
from jax import make_jaxpr
import marimo as mo

# Jax's PyTrees

## Lesson Goals:

By the end of this lesson, you'll know to use the `PyTree` structure and its associated methods to keep your code clean; mastering these concepts and the autodiff will set us up to build our own simple neural network, _a la_  [Equinox](https://github.com/patrick-kidger/equinox)

## Core Concepts:

- What is a `PyTree`?
- what functions does Jax provide to interact with PyTrees?
- Registering `dataclass`-es to leverage the `PyTree` ecosystem, which

## Concepts In action:

In [ ]:
def analyze_pytrees(*args, verbose_tree_print=False):
    for tree_idx, pytree in enumerate(args, start=1):
        leaves = jax.tree.leaves(pytree)
        if verbose_tree_print:
            print(f'Pytree: {repr(pytree):<30}')
        print(f'Tree: {tree_idx}')
        for i, leaf in enumerate(leaves, start=1):
            print(f'\tLeaf #: {i}:')
            print(f'\tLeaf Type: {type(leaf)}')
            if isinstance(leaf, jnp.ndarray) and len(leaf.shape) >= 2:
                builder = '\t'
                for row in leaf:
                    for el in row:
                        builder = builder + str(el) + ','
                    builder = builder + '\n\t\t'
                leaf = builder
            print(f'\tLeaf val: {leaf}')
            print('\t' + '*' * 20)
        print('\n')

# What is a PyTree

A PyTree is a nested structure composed of objects. Much like a tree (in computer science) it can be broken down into nodes: leaf and non-leaf nodes. A leaf node can be thought of as a container that has been "registered" with `Jax` and cannot be deconstructed more, while a leaf is anything else; `Jax` deconstructs things like lists and dictionaries into individual elements, but keeps strings as they are. Here are some practical examples that should cover most data types you would use with `Jax`

Let's take a look at some samples. Let's count how many leaves there are here:

```python
conv1 = {
    "device": "CPU",
    "kernel": jnp.asarray(np.eye(5)),
    "bias": jnp.asarray(np.ones(5)),
    "indices": [1,2,"3"]
}
```

and here:

```python
conv2 = {
    "device": "GPU",
    "kernel": jnp.asarray(np.eye(5)),
    "bias": jnp.asarray(np.ones(5)),
    "metadata": {"gpu": 0, "dtype": jnp.float64}
}
```

In [ ]:
dense1 = {
    "device": "CPU",
    "W": jnp.asarray(np.eye(5)),
    "b": jnp.asarray(np.ones(5)),
}

dense2 = {
    "device": "GPU",
    "W": jnp.asarray(np.eye(5)),
    "b": jnp.asarray(np.ones(5)),
    "metadata": {"gpu": 0, "dtype": jnp.float64}
}

## Manually Counting Leaves

Let's first count leaves to get an intuition

In [ ]:
analyze_pytrees(dense1)

In `dense1` we had 6 elements:

- (1)the bias, a jax vector

- (2) the "device", a string

- (3,4,5) the contents of the indices
- (6), the kernel, the jax matrix

Try to reason through `dense2` in the next cell on your own and verify your understanding - it's imperative that things click now before we move on.

In [ ]:
analyze_pytrees(dense2)

## Exploring the `PyTree`

Let's use some built-in functions to explore the `dense2` value

In [ ]:
def list_flattened(tree):
    flat_vals, flat_tree_def = jax.tree.flatten(tree)

    print(flat_vals)
    print(flat_tree_def)

list_flattened(dense2)

In [ ]:
def list_leaves(tree):
    leaf_vals = jax.tree.leaves(tree)

    print(leaf_vals)

list_leaves(dense2)

## Takeaway: Leaves and Flatten

`leaves` and `flatten` both return the flattened values as a list, but `flatten` also returns the tree definition, which we can use to reconstruct the original tree.

**Note**: there are two variants of `leaves` and `flatten`: `x_with_path`, which breaks down the nested structure and generates the traversal path.

## Reconstructing the PyTree

Let's now take a look at using the result of `flatten` to reconstruct the tree

In [ ]:
def reconstruct_tree(in_tree):
    flat_vals, flat_tree_def = jax.tree.flatten(in_tree)
    recreated_tree = jax.tree.unflatten(flat_tree_def, flat_vals)
    print(recreated_tree.keys())
    print(recreated_tree["metadata"])
    print(f"\nWeights: {recreated_tree['W']}")
    print(f"\nBias: {recreated_tree['b']}")

reconstruct_tree(dense2)

# Why would I use a PyTree over dataclasses?

The lazy answer is that you want to use `PyTree` because of how it natively integrates with `Jax`. When we work with ML tasks, we often want to work with the leaves of our data structure, so it makes sense that the `Jax` team created utility functions to work with these structures. To see this, let's first create an example dataclass, working with the `DenseLayer` from before.

*it's actually possible to integrate the two, which we will cover at the very end of this notebook.

## Dataclass and PyTree Definition

In [ ]:
from dataclasses import dataclass

NUM_FEATURES = 4
HIDDEN_DIM = 16

@dataclass
class DenseLayer:
    """
    The equivalent of conv1, structure-wise
    """
    numerical: dict[str, jnp.ndarray]
    metadata: dict[str, str]

    def __init__(self, W, b, metadata):
        self.numerical = {
            "W": W, "b": b
        }
        self.metadata = {k:v for k,v in metadata.items()}

PTDenseLayer = dict[str, 
    jnp.ndarray | 
    dict[str, str | int | jnp.dtype]
]
def conv_pytree_constructor(
    W: jnp.ndarray,
    b: jnp.ndarray,
    metadata: dict[str, int]
) -> PTDenseLayer:
    return {
        "numerical": {
            "W": W,
            "b": b,
        },
        "metadata": {k:v for k,v in metadata.items()}
    }

metadata = {"device": "GPU", "gpu": 0, "dtype": jnp.float64}
dense_pt = conv_pytree_constructor(
    W=jnp.asarray(np.random.rand(NUM_FEATURES, HIDDEN_DIM)),
    b= jnp.asarray([1, 1, 1, 1]),
    metadata=metadata
)

dense_dtc = DenseLayer(W=jnp.asarray(np.random.rand(NUM_FEATURES, HIDDEN_DIM)),
                     b= jnp.asarray([1, 1, 1, 1]), metadata=metadata
                    )

## Associated `PyTree` methods

We've covered a few functions to destructure the reconstruct the `PyTree` so far. Now let's discuss some of the methods in the context of the pre-defined `DenseLayer` and `PTDenseLayer`. If you've done functional programming before, some of these methods will look familiar to you:

```
jax.tree.map
jax.tree.reduce
jax.tree.transpose

jax.tree.map_with_path
jax.tree.leaves_with_path
jax.tree.flatten_with_path
```

For this tutorial we focus on: `map` and `map_with_path`, which are essential for the datascience workflow.

### Map

A tree map works much like a standard map, where we apply a function to every leaf in our tree

#### Simple Map

Let's add two `PTDenseLayer` using

```python
@jax.jit
def add_numerics(v1, v2):
    return v1 + v2
```

Remember that `Jax` will apply this function, `add_numerics`, to all leaf elements. How do we remove the undesired leaves or stop `Jax` from continuing to descend down structures? For example, we don't need it to traverse "metadata" in this case.

#### Reminder:

As a reminder, here are the leaves in our `dense_pt`

In [ ]:
analyze_pytrees(dense_pt)

Looking at the documentation, we see:

```
jax.tree.map(
    f: 'Callable[..., Any]',
    tree: 'Any',
    *rest: 'Any',
    is_leaf: 'Callable[[Any], bool] | None' = None,
) -> 'Any'
```

It's looking like `is_leaf` might be our solution!

> is_leaf: an optionally specified function that will be called at each flattening step. It should return a boolean, which indicates whether the flattening should traverse the current object, or if it should be stopped immediately, with the whole subtree being treated as a leaf

In [ ]:
def add_numerics(v1, v2):
    if isinstance(v1, dict):
        return v1
    return v1 + v2

is_leaf_filter = lambda x: isinstance(x, dict) and "metadata" in x # TODO: define the filter 
analyze_pytrees(dense_pt)
print("\nFINAL")
analyze_pytrees(jax.tree.map(
    # TODO: fill in the args for the `tree.map`
    add_numerics, dense_pt, dense_pt, is_leaf= is_leaf_filter
))

### Map-with-Path

If you look at the previous cell, you'll notice that we had no notion of

### Map with path

In [ ]:
from typing import Any

def join_path(path: tuple):
    return '.'.join(str(key.key) if hasattr(key, 'key') else str(key) for key in path)

def smart_init(path: tuple, params) -> dict:
    """Initialize parameters based on their role in the model."""
    # Initialize the bias to be zeros-like, as in param
    if not isinstance(params, dict):
        return params
    param = params["dense"]
    b = jnp.zeros_like(param["b"])
    if param["init"] == "Xavier":
        # Xavier initialization for weights
        fan_in, fan_out = param["W"].shape
        bound = jnp.sqrt(6.0 / (fan_in + fan_out))
        W = jax.random.uniform(
            jax.random.PRNGKey(42), param["W"].shape, minval=-bound, maxval=bound
        )
    elif param["init"] == "Normal":
        W = jax.random.normal(
            jax.random.PRNGKey(42), param["W"].shape
        )
    else:
        raise ValueError(f"Unrecognized initialization: {param['init']}")
    return {
        "W": W, "b": b, "init": param["init"]
    }

# Example model parameters
params = {
    'network': {
        'metadata': {"device": "GPU", "gpu": 0, "dtype": jnp.float64},
        'layers': {
            0: {'dense': {'W': jnp.ones((128, 256)), "init": "Xavier", 'b': jnp.ones(256)}},
            1: {'dense': {'W': jnp.ones((256, 128)), "init": "Normal", 'b': jnp.ones(128)}}
        }
    }
}

# Apply path-aware initialization
initialized_params = jax.tree.map_with_path(
    smart_init, params, 
    # We stop at the "dense" dictionary. We have 3 keys: "W", "init" and "b"
    is_leaf= lambda x: isinstance(x, dict) and "dense" in x
)

In [ ]:
initialized_params

**But** this is kind of (very) messy. Notice how we are mixing datatypes in that our "layers" have `jax.ndarray`s and also `str` datatypes. This means that we can't `jit` these functions, which isn't a huge issue as this is a one-time cost, but it's still something to keep in mind.

# Registering Custom Objects

Further above we discussed using `dataclass` to clean up our code and give it structure, instead of slicing and moving dictionary values around. How do we do this?